## librerias

In [1]:
import os
import webbrowser
import pandas as pd
import json
import geopandas as gpd
import colorsys
import numpy as np
import http.server
import socketserver
from threading import Thread
import time
import hashlib
import unicodedata

## bases 

In [2]:
usuario = os.getlogin()

In [3]:
b = pd.read_excel(fr"C:\Users\{usuario}\Downloads\UM_IMB_SUS.xlsx",sheet_name="Hoja2")

In [4]:
b = b.drop(index=[0, 2, 5,3,1])

In [5]:
b = b.drop(columns=['Unnamed: 1'])

In [6]:
b = b.rename(columns={
   'CLUES' : 'preguntas',
})

In [7]:
b

,preguntas
4,Equipo de cómputo
6,Banco de altura
7,Banco giratorio
8,Báscula electrónica con estadímetro
9,Báscula pesabebés electrónica
10,Mesa para colocar báscula pesabebés
11,Bote sanitario con pedal
12,Caja portalaminilla de plástico con separadores
13,Carta Snellen con marco
14,Charola de Mayo de acero inoxidable


In [8]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [9]:
base.columns

Index(['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada',
       'poblacion_sin_dh_menos_30', 'poblacion_con_dh_menos_30', 'pob_imo_pct',
       'CONSULTORIOS GENERALES', 'consultorios_generales_habilitados',
       'Equipo de cómputo', 'Banco de altura', 'Banco giratorio',
       'Báscula electrónica  con estadímetro',
       'Báscula pesabebés electrónica ', 'Bote sanitario con pedal',
       'Caja Portalaminilla de plástico con separadores',
       'Carta Snellen con marco',
       'Charola de Mayo de acero inoxidable, Dimensiones: 49x32cm',
       'Cinta métrica', 'Contenedor de jabón líquido',
       'Contenedor de toallas desechables',
       'Contenedor rígido 7.50 a 9.40 ml.',
       'Cubeta de Acero Inoxidable y bolsa', 'Equipo de cómputo.1',
       'Escritorio médico de 150x60x75', 'Esfigmomanómetro ',
       'Espejo vestidor',
       'Espejos graves o vaginales chicos, medianos y grandes',
       'Estadímetro pediátrico',
       'Estetoscopio cápsula doble. 

## back

In [10]:
base = base.drop(columns=['poblacion_sin_dh_menos_30', 'poblacion_con_dh_menos_30', 'pob_imo_pct'])


base.columns = (
    base.columns
        .str.strip()
        .str.lower()
        .map(
            lambda x: ''.join(
                c for c in unicodedata.normalize('NFD', x)
                if unicodedata.category(c) != 'Mn'
            )
        )
        .str.replace(" ", "_", regex=False)
)

In [11]:
base = base.merge(
    clues[["clues_imb", "entidad"]],
    on="clues_imb",
    how="left"
)

In [12]:
base =  base.drop(columns=['categoria_gerencial_ampliada','consultorios_generales', 'consultorios_generales_habilitados'])

In [13]:
base.columns

Index(['clues_imb', 'nombre_de_la_unidad', 'equipo_de_computo',
       'banco_de_altura', 'banco_giratorio',
       'bascula_electronica__con_estadimetro', 'bascula_pesabebes_electronica',
       'bote_sanitario_con_pedal',
       'caja_portalaminilla_de_plastico_con_separadores',
       'carta_snellen_con_marco',
       'charola_de_mayo_de_acero_inoxidable,_dimensiones:_49x32cm',
       'cinta_metrica', 'contenedor_de_jabon_liquido',
       'contenedor_de_toallas_desechables',
       'contenedor_rigido_7.50_a_9.40_ml.',
       'cubeta_de_acero_inoxidable_y_bolsa', 'equipo_de_computo.1',
       'escritorio_medico_de_150x60x75', 'esfigmomanometro', 'espejo_vestidor',
       'espejos_graves_o_vaginales_chicos,_medianos_y_grandes',
       'estadimetro_pediatrico',
       'estetoscopio_capsula_doble._auxiliar_para_realizar_auscultacion',
       'estetoscopio_pinard_o_doppler_fetal_portatil',
       'guarda_de_medicamentos,_materiales_o_instrumental',
       'guardarropa_con_perchero', 'lam

In [14]:
# Obtener entidades únicas
entidades_unicas = sorted(base['entidad'].dropna().unique())

# Identificar columnas de equipamiento (todas excepto las que no son numéricas)
columnas_excluir = ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad']
columnas_equipamiento = [col for col in base.columns if col not in columnas_excluir]

# Obtener unidades por entidad
def get_unidades_por_entidad(entidad):
    """Obtiene todas las unidades de una entidad con sus datos"""
    df_entidad = base[base['entidad'] == entidad]
    return df_entidad.to_dict('records')

In [15]:
# Configuración de colores
COLOR_PRIMARIO = "#FAF2F5"
COLOR_SECUNDARIO = '#AE8640'
COLOR_HBC = "#FDFDFDC0"
COLOR_FONDO = "#235B4E"
COLOR_BORDE = '#7A1737'
COLOR_TEXTO = '#000000'

In [16]:
# Función simple para formatear nombres de columnas
def formatear_nombre(columna):
    nombre = columna.replace('_', ' ')
    nombre = nombre.split('.')[0]
    palabras = nombre.split()
    palabras = [p.capitalize() for p in palabras]
    return ' '.join(palabras)


## front

In [17]:
script_url = "https://script.google.com/macros/s/AKfycbxacT6QcfRHtns3-8pPqdUMctq5lOnJHSakd66Z3-401ODKNScOuEyDYvkBUJsjtW5y/exec"

In [18]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
# ==================== GENERACIÓN DEL HTML ====================
html_content = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Cuestionario de Equipamiento - IMSS Bienestar</title>
    <link href="https://fonts.googleapis.com/css2?family=League+Spartan:wght@400;500;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: "League Spartan", sans-serif;
        }}

        body {{
            background: linear-gradient(135deg, {COLOR_PRIMARIO} 0%, #fff 100%);
            min-height: 100vh;
            padding: 20px;
        }}

        .container {{
            width: 100%;
            max-width: 1400px;
            margin: 0 auto;
        }}

        .menu-container {{
            position: fixed;
            top: 20px;
            right: 20px;
            z-index: 1001;
        }}

        .menu-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 50%;
            width: 50px;
            height: 50px;
            cursor: pointer;
            display: flex;
            flex-direction: column;
            justify-content: center;
            align-items: center;
            gap: 6px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            transition: 0.3s;
        }}

        .menu-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: scale(1.05);
        }}

        .menu-btn span {{
            width: 25px;
            height: 3px;
            background: white;
            border-radius: 3px;
            transition: 0.3s;
        }}

        .menu-panel {{
            position: fixed;
            top: 0;
            right: -400px;
            width: 380px;
            height: 100%;
            background: white;
            box-shadow: -2px 0 10px rgba(0,0,0,0.1);
            z-index: 1002;
            transition: 0.3s;
            overflow-y: auto;
            padding: 80px 25px 25px 25px;
        }}

        .menu-panel.active {{
            right: 0;
        }}

        .menu-overlay {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            z-index: 1001;
            display: none;
        }}

        .menu-overlay.active {{
            display: block;
        }}

        .menu-panel h2 {{
            color: {COLOR_FONDO};
            margin-bottom: 20px;
            font-size: 24px;
            border-bottom: 3px solid {COLOR_SECUNDARIO};
            padding-bottom: 10px;
        }}

        .menu-panel h3 {{
            color: {COLOR_SECUNDARIO};
            margin: 20px 0 10px 0;
            font-size: 18px;
        }}

        .menu-panel p {{
            color: #333;
            line-height: 1.6;
            margin-bottom: 15px;
        }}

        .menu-panel ul, .menu-panel ol {{
            color: #555;
            margin-left: 20px;
            margin-bottom: 15px;
        }}

        .menu-panel li {{
            margin-bottom: 8px;
        }}

        .close-menu {{
            position: absolute;
            top: 20px;
            right: 20px;
            background: none;
            border: none;
            font-size: 30px;
            cursor: pointer;
            color: {COLOR_FONDO};
        }}

        .header {{
            background: {COLOR_FONDO};
            padding: 20px;
            color: white;
            margin-bottom: 30px;
            border-radius: 15px;
            text-align: center;
            position: relative;
        }}

        .header img {{
            height: 50px;
            margin-bottom: 10px;
        }}

        .header h1 {{
            font-size: 24px;
            margin-bottom: 5px;
        }}

        .instrucciones-rapidas {{
            background: white;
            border-radius: 12px;
            padding: 15px 20px;
            margin-bottom: 20px;
            display: flex;
            justify-content: center;
            gap: 30px;
            flex-wrap: wrap;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}

        .instruccion-item {{
            display: flex;
            align-items: center;
            gap: 12px;
            font-size: 14px;
            font-weight: 500;
        }}

        .instruccion-color {{
            width: 24px;
            height: 24px;
            border-radius: 6px;
        }}

        .color-verde {{
            background: #d4edda;
            border: 2px solid #2e7d32;
        }}

        .color-rojo {{
            background: #ffebee;
            border: 2px solid #c62828;
        }}

        .color-azul {{
            background: #e3f2fd;
            border: 2px solid #1565c0;
        }}

        .color-naranja {{
            background: #fff3e0;
            border: 2px solid #e65100;
        }}

        .color-dorado {{
            background: #fff8e1;
            border: 2px solid #f57f17;
        }}

        .estados-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
            gap: 15px;
            max-height: 600px;
            overflow-y: auto;
            padding: 10px;
        }}

        .estado-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 12px;
            padding: 15px 10px;
            color: white;
            font-weight: 600;
            cursor: pointer;
            transition: 0.3s;
            font-size: 14px;
        }}

        .estado-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: translateY(-3px);
        }}

        .modal {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.8);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal.active {{
            display: flex;
        }}
        
        .modal-form.active {{
            display: flex;
        }}

        .modal-content {{
            position: relative;
            background: linear-gradient(135deg, {COLOR_BORDE}dd);
            backdrop-filter: none;
            border-radius: 20px;
            padding: 30px;
            width: 90%;
            max-width: 450px;
            color: white;
            border: 1px solid rgba(255,255,255,0.2);
            animation: slideUp 0.3s;
        }}

        @keyframes slideUp {{
            from {{ transform: translateY(20px); opacity: 0; }}
            to {{ transform: translateY(0); opacity: 1; }}
        }}

        .modal h2 {{
            text-align: center;
            margin-bottom: 10px;
            font-size: 28px;
        }}

        .modal p {{
            text-align: center;
            margin-bottom: 20px;
            opacity: 0.9;
        }}

        .close-btn {{
            position: absolute;
            top: 10px;
            right: 15px;
            background: none;
            border: none;
            font-size: 28px;
            cursor: pointer;
            color: white;
            opacity: 0.8;
            transition: opacity 0.2s;
            line-height: 1;
        }}

        .close-btn:hover {{
            opacity: 1;
        }}

        .modal-form {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal-form.active {{
            display: flex;
        }}

        .modal-form-content {{
            position: relative;
            background: white;
            border-radius: 20px;
            padding: 30px;
            width: 95%;
            max-width: 1400px;
            max-height: 90vh;
            overflow-x: auto;
            overflow-y: auto;
        }}

        .modal-form-content .close-btn {{
            color: #333;
            top: 10px;
            right: 15px;
        }}

        .form-group {{
            margin-bottom: 15px;
        }}

        .form-group label {{
            display: block;
            margin-bottom: 5px;
            font-weight: 600;
        }}

        .form-group input, .form-group select {{
            width: 100%;
            padding: 10px;
            background: rgba(255,255,255,0.2);
            border: 1px solid rgba(255,255,255,0.3);
            border-radius: 8px;
            color: white;
            font-size: 14px;
        }}

        .form-group input::placeholder {{
            color: rgba(255,255,255,0.6);
        }}

        .form-group input:focus {{
            outline: none;
            border-color: white;
            background: rgba(255,255,255,0.25);
        }}

        .help-text {{
            font-size: 12px;
            margin-top: 5px;
            opacity: 0.8;
        }}

        .btn {{
            width: 100%;
            padding: 12px;
            background: white;
            color: {COLOR_FONDO};
            border: none;
            border-radius: 8px;
            font-weight: 700;
            font-size: 16px;
            cursor: pointer;
            transition: 0.3s;
            margin-top: 10px;
        }}

        .btn:hover {{
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.3);
        }}

        .error {{
            color: #ffcccc;
            font-size: 13px;
            margin-top: 10px;
            text-align: center;
            display: none;
        }}

        .table-container {{
            overflow-x: auto;
            overflow-y: auto;
            max-height: 55vh;
            position: relative;
        }}

        .equipamiento-table {{
            width: 100%;
            border-collapse: collapse;
            position: relative;
        }}

        .equipamiento-table th, .equipamiento-table td {{
            border: 1px solid #ddd;
            padding: 10px 12px;
            text-align: center;
            vertical-align: middle;
        }}

        .equipamiento-table th:nth-child(1),
        .equipamiento-table td:nth-child(1) {{
            position: sticky;
            left: 0;
            background-color: {COLOR_FONDO};
            z-index: 10;
            min-width: 280px;
            max-width: 350px;
            text-align: left;
            color: white;
        }}

        .equipamiento-table td:nth-child(1) {{
            background-color: #f0f0f0;
            color: #333;
            font-weight: 500;
        }}

        .equipamiento-table th:nth-child(1) {{
            background-color: {COLOR_FONDO};
            color: white;
            z-index: 20;
        }}

        .equipamiento-table td:nth-child(1)::after,
        .equipamiento-table th:nth-child(1)::after {{
            content: '';
            position: absolute;
            top: 0;
            right: -5px;
            height: 100%;
            width: 5px;
            box-shadow: 2px 0 5px rgba(0,0,0,0.1);
            pointer-events: none;
        }}

        .equipamiento-table th {{
            background-color: {COLOR_FONDO};
            color: white;
            position: sticky;
            top: 0;
            z-index: 15;
            min-width: 120px;
        }}

        .equipamiento-table th:hover {{
            background-color: #1a8fbb !important;
            cursor: pointer;
        }}

        .equipamiento-table tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}

        .equipamiento-table tbody tr:hover {{
            background-color: #e3f2fd !important;
        }}

        .equipamiento-table tbody tr:hover td:first-child {{
            background-color: #bbdef5 !important;
        }}

        .equipamiento-table td:hover {{
            background-color: #fff3e0 !important;
        }}

        .equipamiento-table td:hover .valor-mostrado {{
            transform: scale(1.05);
            box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        }}

        .equipamiento-table tr.fila-guardada-bd {{
            background-color: #d4edda !important;
        }}

        .equipamiento-table tr.fila-guardada-bd:hover {{
            background-color: #c3e6cb !important;
        }}

        .equipamiento-table tr.fila-guardada-bd td:first-child {{
            background-color: #c3e6cb !important;
        }}

        .campo-container {{
            display: flex;
            gap: 8px;
            align-items: center;
            justify-content: center;
            flex-wrap: wrap;
        }}

        .valor-mostrado {{
            display: inline-block;
            background: #e8f5e9;
            color: #2e7d32;
            padding: 5px 10px;
            border-radius: 5px;
            font-weight: bold;
            font-size: 14px;
            min-width: 50px;
            text-align: center;
            cursor: pointer;
            transition: 0.2s;
        }}

        .valor-mostrado:hover {{
            background: #c8e6c9;
            transform: scale(1.02);
        }}

        .valor-vacio {{
            background: #ffebee;
            color: #c62828;
            cursor: pointer;
        }}

        .valor-vacio:hover {{
            background: #ffcdd2;
        }}

        .valor-booleano {{
            background: #e3f2fd;
            color: #1565c0;
            cursor: pointer;
            min-width: 70px;
        }}

        .valor-booleano:hover {{
            background: #bbdefb;
            transform: scale(1.02);
        }}

        .valor-cero {{
            background: #e3f2fd;
            color: #0d47a1;
            border: 2px solid #1565c0;
        }}

        .valor-cero:hover {{
            background: #bbdefb;
            transform: scale(1.02);
        }}

        .valor-guardado-nube {{
            background: #fff8e1;
            color: #f57f17;
            border: 2px solid #f57f17;
            cursor: default;
        }}

        .valor-guardado-nube::after {{
            content: " ✓";
            color: #4CAF50;
            font-weight: bold;
        }}

        .input-edicion {{
            width: 80px;
            padding: 5px;
            border: 2px solid {COLOR_SECUNDARIO};
            border-radius: 4px;
            text-align: center;
            font-size: 14px;
        }}

        .btn-accion {{
            color: white;
            border: none;
            padding: 5px 10px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 12px;
            font-weight: 600;
            transition: 0.3s;
            white-space: nowrap;
            background: #2196F3;
        }}

        .btn-accion:hover {{
            background: #0b7dda;
        }}

        .btn-guardar-individual {{
            background: #FF9800;
            color: white;
            border: none;
            padding: 5px 12px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 11px;
            font-weight: 600;
            transition: 0.3s;
            white-space: nowrap;
            display: none; /* Oculto pero funcional */
        }}

        .btn-guardar-individual:hover {{
            background: #e68900;
            transform: scale(1.02);
        }}

        .btn-guardar-individual:disabled {{
            background: #ccc;
            cursor: not-allowed;
            transform: none;
        }}

        .btn-guardar-individual.guardado {{
            background: #4CAF50;
        }}

        .btn-si {{
            background: #4CAF50;
            color: white;
            border: none;
            padding: 5px 15px;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
        }}

        .btn-si:hover {{
            background: #388E3C;
            transform: scale(1.05);
        }}

        .btn-no {{
            background: #f44336;
            color: white;
            border: none;
            padding: 5px 15px;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
        }}

        .btn-no:hover {{
            background: #c62828;
            transform: scale(1.05);
        }}

        .btn-guardar {{
            background: {COLOR_SECUNDARIO};
            color: white;
            border: none;
            padding: 10px 20px;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
            font-weight: 600;
            transition: 0.3s;
        }}

        .btn-guardar:hover {{
            background: #0d6efd;
            transform: scale(1.02);
        }}

        .btn-guardar:disabled {{
            background: #999;
            cursor: not-allowed;
            transform: none;
        }}

        .btn-cancelar {{
            background: #666;
            color: white;
        }}

        .acciones {{
            display: flex;
            gap: 10px;
            margin-top: 20px;
            justify-content: center;
            flex-wrap: wrap;
        }}

        .progress {{
            margin-bottom: 20px;
            padding: 10px;
            background: #f0f0f0;
            border-radius: 8px;
            color: #333;
        }}

        .badge {{
            display: inline-block;
            padding: 3px 8px;
            border-radius: 12px;
            font-size: 11px;
            font-weight: bold;
        }}
        
        .badge-success {{
            background: #4CAF50;
            color: white;
        }}
        
        .badge-warning {{
            background: #A57F2C;
            color: white;
        }}

        .save-indicator {{
            position: fixed;
            bottom: 20px;
            right: 20px;
            background: #4CAF50;
            color: white;
            padding: 10px 15px;
            border-radius: 8px;
            font-size: 14px;
            opacity: 0;
            transition: opacity 0.3s;
            z-index: 1000;
        }}

        .equipo-tooltip {{
            position: fixed;
            background: #0b5a7c;
            color: white;
            padding: 8px 15px;
            border-radius: 8px;
            font-size: 14px;
            font-weight: bold;
            z-index: 2000;
            pointer-events: none;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            white-space: nowrap;
            font-family: "League Spartan", sans-serif;
        }}

        .clues-selector {{
            margin-bottom: 20px;
            padding: 15px;
            background: #f8f9fa;
            border-radius: 8px;
            display: flex;
            align-items: center;
            gap: 15px;
            flex-wrap: wrap;
            transition: all 0.3s;
        }}

        .clues-selector.bloqueado {{
            background: #e8e8e8;
            opacity: 0.7;
        }}

        .clues-selector.bloqueado select {{
            background: #d0d0d0;
            cursor: not-allowed;
            pointer-events: none;
        }}

        .clues-selector.bloqueado button {{
            background: #999;
            cursor: not-allowed;
            pointer-events: none;
        }}

        .clues-selector label {{
            font-weight: 600;
            margin-right: 10px;
        }}

        .clues-selector select {{
            padding: 8px 15px;
            border-radius: 5px;
            border: 1px solid #ddd;
            font-size: 14px;
            min-width: 300px;
            background: white;
            flex: 1;
            transition: all 0.3s;
        }}

        .clues-selector button {{
            padding: 8px 20px;
            background: {COLOR_SECUNDARIO};
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
        }}

        .clues-selector button:hover:not(:disabled) {{
            background: #0d6efd;
            transform: scale(1.02);
        }}

        .clues-selector .bloqueo-info {{
            display: none;
            font-size: 13px;
            color: #856404;
            background: #fff3cd;
            padding: 5px 15px;
            border-radius: 5px;
            border: 1px solid #ffeeba;
            font-weight: 500;
        }}

        .clues-selector.bloqueado .bloqueo-info {{
            display: inline-block;
        }}

        .btn-desbloquear {{
            background: #28a745;
            color: white;
            border: none;
            padding: 8px 20px;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
            display: none;
        }}

        .btn-desbloquear:hover {{
            background: #218838;
            transform: scale(1.02);
        }}

        .btn-desbloquear.visible {{
            display: inline-block;
        }}

        .consultorios-config {{
            background: #E6D194;
            padding: 15px 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            border-left: 4px solid #A57F2C;
            display: flex;
            align-items: center;
            gap: 20px;
            flex-wrap: wrap;
        }}

        .consultorios-config label {{
            font-weight: 600;
            color: #333;
        }}

        .consultorios-config input {{
            width: 80px;
            padding: 8px 12px;
            border: 2px solid #A57F2C;
            border-radius: 5px;
            font-size: 16px;
            text-align: center;
        }}

        .consultorios-config button {{
            padding: 8px 20px;
            background: #A57F2C;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
        }}

        .consultorios-config button:hover {{
            background: #e68900;
            transform: scale(1.02);
        }}

        .consultorios-config .info-text {{
            color: #666;
            font-size: 13px;
        }}

        .unidad-info {{
            background: #e3f2fd;
            padding: 12px 18px;
            border-radius: 8px;
            margin-bottom: 15px;
            border-left: 4px solid {COLOR_FONDO};
            display: flex;
            flex-wrap: wrap;
            gap: 20px;
            align-items: center;
        }}

        .unidad-info-item {{
            display: flex;
            align-items: center;
            gap: 8px;
        }}

        .unidad-info-item strong {{
            color: {COLOR_FONDO};
        }}

        .equipo-nombre {{
            font-size: 13px;
            line-height: 1.3;
        }}

        .popup-unidad {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.6);
            backdrop-filter: blur(3px);
            display: none;
            justify-content: center;
            align-items: center;
            z-index: 3000;
        }}

        .popup-unidad.active {{
            display: flex;
        }}

        .popup-content {{
            background: white;
            border-radius: 15px;
            padding: 30px;
            max-width: 500px;
            width: 90%;
            box-shadow: 0 10px 40px rgba(0,0,0,0.3);
            animation: slideUp 0.3s;
        }}

        .popup-content h3 {{
            color: {COLOR_FONDO};
            margin-bottom: 15px;
            text-align: center;
        }}

        .popup-content p {{
            margin: 8px 0;
            color: #333;
        }}

        .popup-content .popup-close {{
            margin-top: 20px;
            padding: 10px 30px;
            background: {COLOR_FONDO};
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-weight: 600;
            width: 100%;
            transition: 0.3s;
        }}

        .popup-content .popup-close:hover {{
            background: {COLOR_SECUNDARIO};
        }}

        .preguntas-individuales {{
            background: #f5f5f5;
            padding: 15px 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            border-left: 4px solid {COLOR_FONDO};
            display: flex;
            flex-direction: column;
            gap: 15px;
        }}

        .pregunta-individual {{
            display: flex;
            align-items: center;
            gap: 20px;
            flex-wrap: wrap;
            padding: 10px;
            background: white;
            border-radius: 8px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}

        .pregunta-individual label {{
            font-weight: 600;
            color: #333;
            min-width: 200px;
            font-size: 15px;
        }}

        .pregunta-individual .btn-group {{
            display: flex;
            gap: 10px;
        }}

        .pregunta-individual .btn-internet {{
            padding: 8px 25px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            font-size: 14px;
            transition: 0.3s;
        }}

        .pregunta-individual .btn-internet-si {{
            background: #e0e0e0;
            color: #333;
        }}

        .pregunta-individual .btn-internet-si:hover {{
            background: #4CAF50;
            color: white;
            transform: scale(1.05);
        }}

        .pregunta-individual .btn-internet-si.active {{
            background: #2e7d32;
            color: white;
            box-shadow: 0 0 0 2px #4CAF50;
        }}

        .pregunta-individual .btn-internet-no {{
            background: #e0e0e0;
            color: #333;
        }}

        .pregunta-individual .btn-internet-no:hover {{
            background: #f44336;
            color: white;
            transform: scale(1.05);
        }}

        .pregunta-individual .btn-internet-no.active {{
            background: #c62828;
            color: white;
            box-shadow: 0 0 0 2px #f44336;
        }}

        .pregunta-individual .internet-status {{
            font-weight: 600;
            padding: 5px 15px;
            border-radius: 20px;
            background: #fff;
            font-size: 14px;
        }}

        .pregunta-individual .internet-status.si {{
            color: #2e7d32;
            background: #c8e6c9;
        }}

        .pregunta-individual .internet-status.no {{
            color: #c62828;
            background: #ffcdd2;
        }}

        .pregunta-individual .internet-status.pendiente {{
            color: #666;
            background: #e0e0e0;
        }}

        .pregunta-individual .input-numero {{
            width: 100px;
            padding: 8px 12px;
            border: 2px solid #ddd;
            border-radius: 5px;
            font-size: 16px;
            text-align: center;
            transition: 0.3s;
        }}

        .pregunta-individual .input-numero:focus {{
            border-color: {COLOR_SECUNDARIO};
            outline: none;
        }}

        .pregunta-individual .input-numero:disabled {{
            background: #f5f5f5;
            cursor: not-allowed;
        }}

        .pregunta-individual .btn-aplicar-numero {{
            padding: 8px 20px;
            background: {COLOR_SECUNDARIO};
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
            display: none; /* Oculto pero funcional */
        }}

        .pregunta-individual .btn-aplicar-numero:hover {{
            background: #0d6efd;
            transform: scale(1.02);
        }}

        .pregunta-individual .btn-guardar-individual-pregunta {{
            padding: 8px 20px;
            background: #FF9800;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            font-size: 13px;
            transition: 0.3s;
            display: none; /* Oculto pero funcional */
        }}

        .pregunta-individual .btn-guardar-individual-pregunta:hover {{
            background: #e68900;
            transform: scale(1.02);
        }}

        .pregunta-individual .btn-guardar-individual-pregunta.guardado {{
            background: #4CAF50;
        }}

        .pregunta-individual .btn-guardar-individual-pregunta:disabled {{
            background: #ccc;
            cursor: not-allowed;
        }}

        .pregunta-individual .valor-actual {{
            font-weight: 600;
            padding: 5px 15px;
            border-radius: 20px;
            background: #e8f5e9;
            color: #2e7d32;
            font-size: 14px;
        }}

        .pregunta-individual .valor-actual.vacio {{
            background: #ffebee;
            color: #c62828;
        }}

        .pregunta-individual .valor-actual.cero {{
            background: #e3f2fd;
            color: #0d47a1;
            border: 2px solid #1565c0;
        }}

        .pregunta-individual .valor-actual.guardado-nube {{
            background: #fff8e1;
            color: #f57f17;
            border: 2px solid #f57f17;
        }}

        .loading-spinner {{
            display: none;
            text-align: center;
            padding: 20px;
        }}

        .loading-spinner.active {{
            display: block;
        }}

        .spinner {{
            border: 4px solid #f3f3f3;
            border-top: 4px solid {COLOR_FONDO};
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 0 auto;
        }}

        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}

        .notificacion {{
            padding: 10px 20px;
            border-radius: 8px;
            margin: 10px 0;
            font-weight: 500;
            display: none;
        }}

        .notificacion.info {{
            background: #d1ecf1;
            color: #0c5460;
            border: 1px solid #bee5eb;
        }}

        .notificacion.success {{
            background: #d4edda;
            color: #155724;
            border: 1px solid #c3e6cb;
        }}

        .notificacion.warning {{
            background: #fff3cd;
            color: #856404;
            border: 1px solid #ffeeba;
        }}

        .notificacion.error {{
            background: #f8d7da;
            color: #721c24;
            border: 1px solid #f5c6cb;
        }}
    </style>
</head>
<body>
    <div class="menu-container">
        <button class="menu-btn" onclick="toggleMenu()">
            <span></span>
            <span></span>
            <span></span>
        </button>
    </div>

    <div class="menu-overlay" id="menuOverlay" onclick="toggleMenu()"></div>
    
    <div class="menu-panel" id="menuPanel">
        <button class="close-menu" onclick="toggleMenu()">&times;</button>
        <h2>Instrucciones</h2>
        
        <h3>1. Seleccionar Estado</h3>
        <p>Haga clic en el boton del estado correspondiente para comenzar el registro de equipamiento.</p>
        
        <h3>2. Registrar Datos del Usuario</h3>
        <p>Complete el formulario con nombre y correo electronico.</p>
        
        <h3>3. Seleccionar Unidad Medica</h3>
        <p>Elija la unidad medica especifica que desea registrar en el selector desplegable.</p>
        
        <h3>4. Configurar Consultorios</h3>
        <p>Ingrese el numero de consultorios que tiene la unidad y presione "Aplicar".</p>
        
        <h3>5. Preguntas Individuales</h3>
        <p>Responda las preguntas que no dependen de consultorios:</p>
        <ul>
            <li><strong>Internet:</strong> Seleccione SI o NO (se guarda automáticamente)</li>
            <li><strong>Consultorios Generales Habilitados:</strong> Ingrese el número y presione Enter (se guarda automáticamente)</li>
        </ul>
        
        <h3>6. Llenar Cuestionario</h3>
        <p>Haga clic en cualquier celda para editarla, ingrese el valor y presione <strong>Enter</strong> para guardar automáticamente.</p>
        <p><strong style="color:#2e7d32">VERDE:</strong> Valor > 0 registrado</p>
        <p><strong style="color:#0d47a1">AZUL:</strong> Valor 0 registrado (válido)</p>
        <p><strong style="color:#c62828">ROJO:</strong> Campo pendiente - Haga clic para llenar</p>
        <p><strong style="color:#f57f17">DORADO:</strong> Valor guardado en la nube ✓</p>
        <p><strong style="color:#d4edda">FILA VERDE:</strong> Unidad ya guardada en la base de datos</p>
        
        <h3>7. Guardado Automático</h3>
        <p>Cada pregunta se guarda automáticamente al presionar <strong>Enter</strong> después de ingresar el valor.</p>
        
        <h3>8. Ver Detalles</h3>
        <p>Haga clic en <strong>"Ver detalles"</strong> para mostrar informacion de la unidad.</p>
        
        <h3>Consejos</h3>
        <ul>
            <li>Use solo numeros enteros para las cantidades</li>
            <li>El <strong>0</strong> es un valor válido y se puede guardar</li>
            <li>Cada campo se guarda automáticamente al presionar Enter</li>
            <li>El progreso local se guarda automaticamente</li>
            <li>Pase el mouse sobre cualquier celda para ver que pregunta esta respondiendo</li>
            <li>Busque el icono ✓ para saber que un valor ya fue guardado en la nube</li>
        </ul>
        
        <h3>Soporte</h3>
        <p>Si tiene problemas, contacte al area de sistemas de IMSS Bienestar.</p>
    </div>

    <div class="container">
        <div class="header">
            <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" alt="IMSS Bienestar">
            <h1>CUESTIONARIO DE EQUIPAMIENTO POR UNIDAD MEDICA</h1>
            <p>Seleccione una entidad para registrar el equipamiento de sus unidades</p>
        </div>

        <div id="loadingEstados" class="loading-spinner">
            <div class="spinner"></div>
            <p>Cargando estados...</p>
        </div>

        <div class="estados-grid" id="estadosGrid">
'''

# Generar botones de entidades
for entidad in entidades_unicas:
    num_unidades = len(base[base['entidad'] == entidad])
    html_content += f'''
            <button class="estado-btn" onclick="abrirModal('{entidad}')">
                {entidad}<br>
                <small style="font-size: 11px;">{""}</small>
            </button>
    '''

html_content += f'''
        </div>
    </div>

    <div class="modal" id="loginModal">
        <div class="modal-content">
            <button class="close-btn" onclick="cerrarModal()">&times;</button>
            <h2 id="modalEstado"></h2>
            <p>Registre sus datos para continuar</p>
            
            <div class="form-group">
                <label>Entidad</label>
                <input type="text" id="usuarioInput" readonly>
            </div>
            
            <div class="form-group">
                <label>Nombre completo</label>
                <input type="text" id="nombreInput" placeholder="Ej: Juan Carlos Perez Gonzalez" required>
                <div class="help-text">Ingrese nombre(s) y apellidos completos</div>
            </div>
            
            <div class="form-group">
                <label>Correo electronico institucional</label>
                <input type="email" id="emailInput" placeholder="ejemplo@imssbienestar.gob.mx" required>
                <div class="help-text">Ingrese su correo electronico</div>
            </div>
            
            <button class="btn" onclick="validarDatos()">Continuar</button>
            <div id="errorMsg" class="error"></div>
        </div>
    </div>

    <div class="popup-unidad" id="popupUnidad">
        <div class="popup-content">
            <h3>Detalles de la Unidad</h3>
            <div id="popupInfo">
                <p><strong>CLUES:</strong> <span id="popupClues"></span></p>
                <p><strong>Unidad Medica:</strong> <span id="popupNombre"></span></p>
                <p><strong>Categoria:</strong> <span id="popupCategoria"></span></p>
                <p><strong>Entidad:</strong> <span id="popupEntidad"></span></p>
                <p><strong>Registrado por:</strong> <span id="popupUsuario"></span></p>
                <p><strong>Consultorios:</strong> <span id="popupConsultorios"></span></p>
                <p><strong>Internet:</strong> <span id="popupInternet"></span></p>
                <p><strong>Consultorios Habilitados:</strong> <span id="popupConsultoriosHabilitados"></span></p>
            </div>
            <button class="popup-close" onclick="cerrarPopup()">Cerrar</button>
        </div>
    </div>

    <div class="modal-form" id="formModal">
        <div class="modal-form-content">
            <button class="close-btn" onclick="cerrarFormModal()">&times;</button>
            <h2 id="formEstado"></h2>
            <div id="userInfo" style="background: #f0f0f0; padding: 10px; border-radius: 8px; margin-bottom: 15px; font-size: 14px;"></div>
            
            <div id="notificacion" class="notificacion"></div>
            
            <div class="clues-selector" id="cluesSelectorContainer">
                <label for="cluesSelector">Seleccionar Unidad Medica:</label>
                <select id="cluesSelector" onchange="cambiarUnidadSeleccionada()">
                    <option value="">-- Seleccione una unidad --</option>
                </select>
                <button onclick="cargarUnidades(estadoSeleccionado)">Recargar</button>
                <button onclick="mostrarPopupUnidad()" style="background:#1E5B4F;">Ver detalles</button>
                <button class="btn-desbloquear" id="btnDesbloquear" onclick="desbloquearSelector()">Desbloquear</button>
                <span class="bloqueo-info"><i class="fas fa-lock"></i> Selector bloqueado - Presione "Desbloquear" o "Guardar"</span>
            </div>
            
            <div id="unidadInfo" class="unidad-info" style="display:none;">
                <span class="unidad-info-item"><strong>CLUES:</strong> <span id="cluesUnidadSeleccionada"></span></span>
                <span class="unidad-info-item"><strong>Unidad:</strong> <span id="nombreUnidadSeleccionada"></span></span>
                <span class="unidad-info-item"><strong>Categoria:</strong> <span id="categoriaUnidadSeleccionada"></span></span>
            </div>
            
            <!-- Preguntas Individuales -->
            <div class="preguntas-individuales">
                <!-- Internet -->
                <div class="pregunta-individual">
                    <label><i class="fas fa-wifi"></i> ¿Cuenta con servicio de Internet?</label>
                    <div class="btn-group">
                        <button class="btn-internet btn-internet-si" id="btnInternetSi" onclick="seleccionarInternet(true)">SI</button>
                        <button class="btn-internet btn-internet-no" id="btnInternetNo" onclick="seleccionarInternet(false)">NO</button>
                    </div>
                    <span class="internet-status pendiente" id="internetStatus">PENDIENTE</span>
                    <button class="btn-guardar-individual-pregunta" id="btnGuardarInternet" onclick="guardarPreguntaIndividual('{PREGUNTA_INTERNET}')">Guardar</button>
                </div>
                
                <!-- Consultorios Generales Habilitados -->
                <div class="pregunta-individual">
                    <label><i class="fas fa-hospital"></i> Consultorios Generales Habilitados:</label>
                    <input type="number" class="input-numero" id="inputConsultoriosHabilitados" 
                           min="0" max="50" placeholder="0"
                           onkeypress="if(event.key==='Enter'){{ guardarConsultoriosHabilitados(); guardarPreguntaIndividual('{PREGUNTA_CONSULTORIOS_HABILITADOS}'); }}">
                    <span class="valor-actual vacio" id="consultoriosHabilitadosStatus">PENDIENTE</span>
                    <button class="btn-guardar-individual-pregunta" id="btnGuardarHabilitados" onclick="guardarPreguntaIndividual('{PREGUNTA_CONSULTORIOS_HABILITADOS}')">Guardar</button>
                </div>
            </div>
            
            <div class="consultorios-config">
                <label for="numConsultorios">Numero de consultorios:</label>
                <input type="number" id="numConsultorios" min="0" max="20" value="1">
                <button onclick="aplicarConsultorios()">Aplicar</button>
                <span class="info-text" id="consultoriosInfo">Consultorios configurados: 1</span>
            </div>
            
            <div id="progressInfo" class="progress"></div>
            
            <div class="instrucciones-rapidas">
                <div class="instruccion-item">
                    <div class="instruccion-color color-verde"></div>
                    <div class="instruccion-texto"><strong>VERDE</strong> Valor > 0 registrado</div>
                </div>
                <div class="instruccion-item">
                    <div class="instruccion-color color-azul"></div>
                    <div class="instruccion-texto"><strong>AZUL</strong> Valor 0 registrado (válido)</div>
                </div>
                <div class="instruccion-item">
                    <div class="instruccion-color color-rojo"></div>
                    <div class="instruccion-texto"><strong>ROJO</strong> Campo pendiente</div>
                </div>
                <div class="instruccion-item">
                    <div class="instruccion-color color-dorado"></div>
                    <div class="instruccion-texto"><strong>DORADO</strong> Valor guardado en nube ✓</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #d4edda; width: 24px; height: 24px; border-radius: 6px; border: 1px solid #2e7d32;"></div>
                    <div class="instruccion-texto"><strong>FILA VERDE</strong> Unidad ya guardada</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #2196F3; width: 24px; height: 24px; border-radius: 6px;"></div>
                    <div class="instruccion-texto"><strong>EDITAR</strong> Click en la celda para editar</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #4CAF50; width: 24px; height: 24px; border-radius: 6px;"></div>
                    <div class="instruccion-texto"><strong>ENTER</strong> Guarda automáticamente</div>
                </div>
            </div>
            
            <div class="table-container">
                <table class="equipamiento-table" id="equipamientoTable">
                    <thead id="tableHead">
                        <tr>
                            <th style="min-width: 280px; max-width: 350px;">Pregunta</th>
                        </tr>
                    </thead>
                    <tbody id="tableBody">
                    </tbody>
                </table>
            </div>
            
            <div class="acciones">
                <button type="button" class="btn btn-cancelar" onclick="cerrarFormModal()">Cerrar</button>
            </div>
        </div>
    </div>

    <div class="save-indicator" id="saveIndicator">
        Pregunta guardada
    </div>

    <script>
        // ==================== VARIABLES GLOBALES ====================
        let celdaEnEdicion = null;
        let tooltip = null;
        let filasGuardadasBD = new Set();
        let unidadSeleccionada = null;
        let numConsultorios = 1;
        let estadoSeleccionado = '';
        let datosActuales = [];
        let selectorBloqueado = false;
        let usuarioActual = {{
            nombre: '',
            email: '',
            entidad: ''
        }};
        let camposGuardadosNube = new Set();

        // Datos desde Python
        const columnasEquipamiento = {json.dumps(columnas_equipamiento)};
        const PREGUNTA_INTERNET = "{PREGUNTA_INTERNET}";
        const PREGUNTA_CONSULTORIOS_HABILITADOS = "{PREGUNTA_CONSULTORIOS_HABILITADOS}";
        const datosUnidades = {json.dumps({entidad: get_unidades_por_entidad(entidad) for entidad in entidades_unicas}, ensure_ascii=False)};
        const SCRIPT_URL = '{script_url}';

        // ==================== FUNCIONES DE BLOQUEO DEL SELECTOR ====================
        function bloquearSelector() {{
            selectorBloqueado = true;
            document.getElementById('cluesSelectorContainer').classList.add('bloqueado');
            document.getElementById('btnDesbloquear').classList.add('visible');
            mostrarNotificacion('Selector de unidades bloqueado. Presione "Desbloquear" para cambiar de unidad.', 'warning');
        }}

        function desbloquearSelector() {{
            selectorBloqueado = false;
            document.getElementById('cluesSelectorContainer').classList.remove('bloqueado');
            document.getElementById('btnDesbloquear').classList.remove('visible');
            mostrarNotificacion('Selector de unidades desbloqueado', 'info');
        }}

        // ==================== FUNCIONES DE UTILIDAD ====================
        function formatearNombreEquipo(nombre) {{
            if (!nombre) return '';
            let formateado = nombre.replace(/_/g, ' ');
            formateado = formateado.split(' ').map(palabra => 
                palabra.charAt(0).toUpperCase() + palabra.slice(1).toLowerCase()
            ).join(' ');
            return formateado;
        }}

        function formatearNombre(nombre) {{
            if (!nombre) return '';
            let formateado = nombre.replace(/_/g, ' ');
            formateado = formateado.split(' ').map(palabra => 
                palabra.charAt(0).toUpperCase() + palabra.slice(1).toLowerCase()
            ).join(' ');
            return formateado;
        }}

        function crearTooltip() {{
            if (!tooltip) {{
                tooltip = document.createElement('div');
                tooltip.className = 'equipo-tooltip';
                tooltip.style.display = 'none';
                document.body.appendChild(tooltip);
            }}
            return tooltip;
        }}

        function mostrarTooltip(event, texto) {{
            const tooltip = crearTooltip();
            tooltip.textContent = texto;
            tooltip.style.display = 'block';
            let left = event.pageX + 15;
            let top = event.pageY - 30;
            
            if (left + tooltip.offsetWidth > window.innerWidth) {{
                left = event.pageX - tooltip.offsetWidth - 15;
            }}
            if (top < 0) {{
                top = event.pageY + 20;
            }}
            
            tooltip.style.left = left + 'px';
            tooltip.style.top = top + 'px';
        }}

        function ocultarTooltip() {{
            if (tooltip) {{
                tooltip.style.display = 'none';
            }}
        }}

        function toggleMenu() {{
            const panel = document.getElementById('menuPanel');
            const overlay = document.getElementById('menuOverlay');
            panel.classList.toggle('active');
            overlay.classList.toggle('active');
        }}

        function mostrarNotificacion(mensaje, tipo = 'info') {{
            const notificacion = document.getElementById('notificacion');
            if (notificacion) {{
                notificacion.textContent = mensaje;
                notificacion.className = 'notificacion ' + tipo;
                notificacion.style.display = 'block';
                
                setTimeout(() => {{
                    notificacion.style.display = 'none';
                }}, 5000);
            }}
        }}

        function mostrarCargando(mostrar) {{
            const spinner = document.getElementById('loadingEstados');
            if (spinner) {{
                spinner.classList.toggle('active', mostrar);
            }}
        }}

        // ==================== FUNCIONES DE MENÚ Y MODALES ====================
        document.addEventListener('keydown', function(e) {{
            if (e.key === 'Escape') {{
                const panel = document.getElementById('menuPanel');
                const overlay = document.getElementById('menuOverlay');
                panel.classList.remove('active');
                overlay.classList.remove('active');
                if (celdaEnEdicion === null) {{
                    cerrarModal();
                    cerrarFormModal();
                    cerrarPopup();
                }}
                ocultarTooltip();
            }}
        }});

        function abrirModal(estado) {{
            estadoSeleccionado = estado;
            document.getElementById('modalEstado').textContent = estado;
            document.getElementById('usuarioInput').value = estado;
            document.getElementById('nombreInput').value = '';
            document.getElementById('emailInput').value = '';
            document.getElementById('errorMsg').style.display = 'none';
            document.getElementById('loginModal').classList.add('active');
            document.getElementById('nombreInput').focus();
        }}

        function cerrarModal() {{
            document.getElementById('loginModal').classList.remove('active');
        }}

        function validarNombreCompleto(nombre) {{
            const nombreTrim = nombre.trim();
            const partes = nombreTrim.split(/\\s+/);
            if (partes.length < 2) return false;
            if (partes.length > 5) return false;
            for (let parte of partes) {{
                if (parte.length < 2) return false;
            }}
            return true;
        }}

        function validarDatos() {{
            const nombre = document.getElementById('nombreInput').value;
            const email = document.getElementById('emailInput').value.trim();
            
            if (!validarNombreCompleto(nombre)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese su nombre completo';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            if (email === '') {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            const emailRegex = /^[^\\s@]+@([^\\s@]+\\.)+[^\\s@]+$/;
            if (!emailRegex.test(email)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico valido';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            usuarioActual = {{
                nombre: nombre.trim(),
                email: email,
                entidad: estadoSeleccionado
            }};
            
            document.getElementById('errorMsg').style.display = 'none';
            cerrarModal();
            abrirFormulario(estadoSeleccionado);
        }}

        function cerrarFormModal() {{
            if (celdaEnEdicion !== null) {{
                alert('Primero presione Enter para guardar el valor que esta editando');
                return;
            }}
            document.getElementById('formModal').classList.remove('active');
            ocultarTooltip();
        }}

        function cerrarPopup() {{
            document.getElementById('popupUnidad').classList.remove('active');
        }}

        // ==================== FUNCIONES DE GUARDADO INDIVIDUAL ====================
        function guardarPreguntaIndividual(preguntaKey) {{
            if (!unidadSeleccionada) {{
                alert('Primero seleccione una unidad medica');
                return;
            }}
            
            let valor = null;
            let nombrePregunta = preguntaKey;
            
            if (preguntaKey === PREGUNTA_INTERNET) {{
                valor = unidadSeleccionada[PREGUNTA_INTERNET];
                if (valor === undefined || valor === null) {{
                    alert('Primero seleccione SI o NO para Internet');
                    return;
                }}
                nombrePregunta = 'Internet';
            }} else if (preguntaKey === PREGUNTA_CONSULTORIOS_HABILITADOS) {{
                const input = document.getElementById('inputConsultoriosHabilitados');
                let num = parseInt(input.value);
                if (isNaN(num) || num < 0) {{
                    alert('Ingrese un numero valido para Consultorios Habilitados');
                    return;
                }}
                if (num > 50) num = 50;
                valor = num;
                unidadSeleccionada[PREGUNTA_CONSULTORIOS_HABILITADOS] = num;
                nombrePregunta = 'Consultorios Habilitados';
            }} else {{
                valor = unidadSeleccionada[preguntaKey];
                if (valor === undefined || valor === null || valor === '') {{
                    alert('Primero ingrese un valor para esta pregunta');
                    return;
                }}
                nombrePregunta = formatearNombreEquipo(preguntaKey);
            }}
            
            const datosGuardar = {{
                entidad: estadoSeleccionado,
                usuario_nombre: usuarioActual.nombre,
                usuario_email: usuarioActual.email,
                clues_imb: unidadSeleccionada.clues_imb,
                nombre_de_la_unidad: unidadSeleccionada.nombre_de_la_unidad,
                categoria: unidadSeleccionada.categoria_gerencial_ampliada,
                num_consultorios: numConsultorios,
                pregunta: preguntaKey,
                valor: valor
            }};
            
            if (preguntaKey.includes('_consultorio_')) {{
                const partes = preguntaKey.split('_consultorio_');
                if (partes.length === 2) {{
                    datosGuardar.pregunta_base = partes[0];
                    datosGuardar.num_consultorio = parseInt(partes[1]);
                }}
            }}
            
            const btn = document.getElementById('btnGuardar' + preguntaKey.replace(/[^a-zA-Z0-9]/g, ''));
            if (btn) {{
                btn.disabled = true;
                btn.textContent = 'Guardando...';
            }}
            
            fetch(SCRIPT_URL, {{ 
                method: 'POST', 
                mode: 'no-cors',
                headers: {{ 'Content-Type': 'application/json' }},
                body: JSON.stringify(datosGuardar)
            }})
            .then(() => {{
                const claveNube = usuarioActual.email + '|' + unidadSeleccionada.clues_imb + '|' + preguntaKey;
                camposGuardadosNube.add(claveNube);
                
                if (btn) {{
                    btn.disabled = false;
                    btn.textContent = '✓ Guardado';
                    btn.classList.add('guardado');
                    // Ocultar después de 2 segundos
                    setTimeout(() => {{
                        btn.textContent = 'Guardar';
                        btn.classList.remove('guardado');
                    }}, 2000);
                }}
                
                const indicator = document.getElementById('saveIndicator');
                indicator.textContent = '✓ Pregunta guardada: ' + nombrePregunta;
                indicator.style.opacity = '1';
                
                mostrarNotificacion('Pregunta "' + nombrePregunta + '" guardada exitosamente', 'success');
                actualizarEstadoGuardado(preguntaKey);
                
                setTimeout(() => {{
                    indicator.style.opacity = '0';
                }}, 2000);
            }})
            .catch(error => {{
                console.error('Error al guardar:', error);
                if (btn) {{
                    btn.disabled = false;
                    btn.textContent = 'Guardar';
                }}
                mostrarNotificacion('Error al guardar la pregunta', 'error');
            }});
        }}

        function actualizarEstadoGuardado(preguntaKey) {{
            const rows = document.querySelectorAll('#equipamientoTable tbody tr');
            for (const row of rows) {{
                const cells = row.cells;
                if (cells.length > 0) {{
                    const preguntaTexto = cells[0].textContent.trim();
                    const preguntaKeyRow = preguntaTexto.toLowerCase().replace(/ /g, '_');
                    
                    for (let j = 1; j < cells.length; j++) {{
                        const colKey = preguntaKeyRow + '_consultorio_' + j;
                        if (colKey === preguntaKey) {{
                            const container = cells[j].querySelector('.campo-container');
                            if (container) {{
                                const valorMostrado = container.querySelector('.valor-mostrado');
                                if (valorMostrado && !valorMostrado.classList.contains('valor-guardado-nube')) {{
                                    valorMostrado.classList.add('valor-guardado-nube');
                                }}
                            }}
                        }}
                    }}
                }}
            }}
            
            if (preguntaKey === PREGUNTA_INTERNET) {{
                const status = document.getElementById('internetStatus');
                if (status) {{
                    status.classList.add('guardado-nube');
                }}
            }} else if (preguntaKey === PREGUNTA_CONSULTORIOS_HABILITADOS) {{
                const status = document.getElementById('consultoriosHabilitadosStatus');
                if (status) {{
                    status.classList.add('guardado-nube');
                }}
            }}
        }}

        // ==================== FUNCIONES DE PRECARGA DE DATOS ====================
        async function consultarDatosCompletos(clues_imb) {{
            try {{
                const response = await fetch(SCRIPT_URL, {{
                    method: 'POST',
                    mode: 'cors',
                    headers: {{
                        'Content-Type': 'application/json',
                    }},
                    body: JSON.stringify({{
                        accion: 'consultarDatosCompletos',
                        clues_imb: clues_imb
                    }})
                }});

                const data = await response.json();
                return data;
            }} catch (error) {{
                console.error('Error al consultar datos:', error);
                return null;
            }}
        }}

        async function precargarDatosUnidad(clues) {{
            try {{
                mostrarCargando(true);
                const data = await consultarDatosCompletos(clues);
                
                if (data && data.existe) {{
                    if (data.datos_unidad) {{
                        const unidad = data.datos_unidad;
                        
                        document.getElementById('nombreUnidadSeleccionada').textContent = 
                            unidad.nombre_de_la_unidad || 'Sin nombre';
                        document.getElementById('cluesUnidadSeleccionada').textContent = 
                            unidad.clues_imb || 'Sin CLUES';
                        document.getElementById('categoriaUnidadSeleccionada').textContent = 
                            unidad.categoria_gerencial_ampliada || 'Sin categoria';
                        
                        const internetValor = unidad.internet;
                        if (internetValor === 'SI' || internetValor === true) {{
                            seleccionarInternet(true);
                            const claveNube = usuarioActual.email + '|' + clues + '|' + PREGUNTA_INTERNET;
                            camposGuardadosNube.add(claveNube);
                        }} else if (internetValor === 'NO' || internetValor === false) {{
                            seleccionarInternet(false);
                            const claveNube = usuarioActual.email + '|' + clues + '|' + PREGUNTA_INTERNET;
                            camposGuardadosNube.add(claveNube);
                        }}
                        
                        const habilitados = parseInt(unidad.consultorios_habilitados);
                        if (!isNaN(habilitados)) {{
                            document.getElementById('inputConsultoriosHabilitados').value = habilitados;
                            unidadSeleccionada[PREGUNTA_CONSULTORIOS_HABILITADOS] = habilitados;
                            const status = document.getElementById('consultoriosHabilitadosStatus');
                            if (habilitados === 0) {{
                                status.textContent = '0 habilitados';
                                status.className = 'valor-actual cero';
                            }} else {{
                                status.textContent = habilitados + ' habilitados';
                                status.className = 'valor-actual';
                            }}
                            const claveNube = usuarioActual.email + '|' + clues + '|' + PREGUNTA_CONSULTORIOS_HABILITADOS;
                            camposGuardadosNube.add(claveNube);
                        }}
                        
                        const numCons = parseInt(unidad.num_consultorios);
                        if (!isNaN(numCons) && numCons > 0) {{
                            document.getElementById('numConsultorios').value = numCons;
                            aplicarConsultorios();
                        }}
                    }}
                    
                    if (data.respuestas) {{
                        const rows = document.querySelectorAll('#equipamientoTable tbody tr');
                        for (const row of rows) {{
                            const cells = row.cells;
                            if (cells.length > 1) {{
                                const preguntaTexto = cells[0].textContent.trim();
                                const preguntaKey = preguntaTexto.toLowerCase().replace(/ /g, '_');
                                
                                for (let j = 1; j < cells.length; j++) {{
                                    const consultorioNum = j;
                                    const cellKey = preguntaKey + '_consultorio_' + consultorioNum;
                                    
                                    if (data.respuestas[cellKey] !== undefined) {{
                                        const valor = data.respuestas[cellKey];
                                        const container = cells[j].querySelector('.campo-container');
                                        if (container) {{
                                            const valorMostrado = container.querySelector('.valor-mostrado');
                                            if (valorMostrado && valor !== null && valor !== undefined && valor !== '') {{
                                                valorMostrado.textContent = valor;
                                                valorMostrado.className = 'valor-mostrado';
                                                if (valor === 0) {{
                                                    valorMostrado.classList.add('valor-cero');
                                                }}
                                                valorMostrado.classList.add('valor-guardado-nube');
                                                const claveNube = usuarioActual.email + '|' + clues + '|' + cellKey;
                                                camposGuardadosNube.add(claveNube);
                                                
                                                const filaIndex = Array.from(rows).indexOf(row);
                                                if (datosActuales[filaIndex]) {{
                                                    datosActuales[filaIndex][cellKey] = valor;
                                                }}
                                            }}
                                        }}
                                    }}
                                }}
                            }}
                        }}
                        
                        mostrarNotificacion('Datos precargados correctamente', 'success');
                    }}
                }} else {{
                    mostrarNotificacion('No hay datos previos para este CLUES', 'info');
                }}
                
                return data;
            }} catch (error) {{
                console.error('Error al precargar datos:', error);
                mostrarNotificacion('Error al cargar datos existentes', 'error');
                return null;
            }} finally {{
                mostrarCargando(false);
            }}
        }}

        // ==================== FUNCIONES DEL FORMULARIO ====================
        function abrirFormulario(estado) {{
            document.getElementById('formEstado').innerHTML = 'Cuestionario de Equipamiento - <strong>' + estado + '</strong>';
            document.getElementById('userInfo').innerHTML = 
                '<strong>Registrado por:</strong> ' + usuarioActual.nombre + ' | ' +
                '<strong>Correo:</strong> ' + usuarioActual.email + ' | ' +
                '<strong>Entidad:</strong> ' + usuarioActual.entidad;
            document.getElementById('formModal').classList.add('active');
            
            cargarUnidades(estado);
        }}

        function cargarUnidades(estado) {{
            const unidades = datosUnidades[estado] || [];
            datosActuales = JSON.parse(JSON.stringify(unidades));
            filasGuardadasBD.clear();
            camposGuardadosNube.clear();
            
            const selector = document.getElementById('cluesSelector');
            selector.innerHTML = '<option value="">-- Seleccione una unidad --</option>';
            
            for (let i = 0; i < datosActuales.length; i++) {{
                const unidad = datosActuales[i];
                const option = document.createElement('option');
                option.value = unidad.clues_imb || i;
                const label = unidad.clues_imb + ' - ' + (unidad.nombre_de_la_unidad || 'Sin nombre');
                option.textContent = label.length > 60 ? label.substring(0, 57) + '...' : label;
                selector.appendChild(option);
            }}
            
            unidadSeleccionada = null;
            document.getElementById('unidadInfo').style.display = 'none';
            document.getElementById('tableBody').innerHTML = '';
            document.getElementById('progressInfo').innerHTML = '<p>Seleccione una unidad medica para comenzar</p>';
            resetearPreguntasIndividuales();
            
            if (selectorBloqueado) desbloquearSelector();
            
            mostrarNotificacion('Seleccione una unidad medica para comenzar', 'info');
        }}

        async function cambiarUnidadSeleccionada() {{
            const selector = document.getElementById('cluesSelector');
            const valorSeleccionado = selector.value;
            
            if (!valorSeleccionado) {{
                unidadSeleccionada = null;
                document.getElementById('unidadInfo').style.display = 'none';
                document.getElementById('tableBody').innerHTML = '';
                document.getElementById('progressInfo').innerHTML = '<p>Seleccione una unidad medica para comenzar</p>';
                resetearPreguntasIndividuales();
                camposGuardadosNube.clear();
                if (selectorBloqueado) desbloquearSelector();
                return;
            }}
            
            unidadSeleccionada = datosActuales.find(u => u.clues_imb === valorSeleccionado) || 
                               datosActuales[parseInt(valorSeleccionado)];
            
            if (unidadSeleccionada) {{
                mostrarUnidadSeleccionada();
                cargarPreguntasIndividuales();
                
                bloquearSelector();
                
                if (unidadSeleccionada.clues_imb) {{
                    await precargarDatosUnidad(unidadSeleccionada.clues_imb);
                }}
            }}
        }}

        function mostrarUnidadSeleccionada() {{
            if (!unidadSeleccionada) return;
            
            document.getElementById('nombreUnidadSeleccionada').textContent = unidadSeleccionada.nombre_de_la_unidad || 'Sin nombre';
            document.getElementById('cluesUnidadSeleccionada').textContent = unidadSeleccionada.clues_imb || 'Sin CLUES';
            document.getElementById('categoriaUnidadSeleccionada').textContent = unidadSeleccionada.categoria_gerencial_ampliada || 'Sin categoria';
            document.getElementById('unidadInfo').style.display = 'flex';
            
            aplicarConsultorios();
        }}

        // ==================== FUNCIONES PARA PREGUNTAS INDIVIDUALES ====================
        function seleccionarInternet(valor) {{
            if (!unidadSeleccionada) {{
                alert('Primero seleccione una unidad medica');
                return;
            }}
            
            unidadSeleccionada[PREGUNTA_INTERNET] = valor;
            
            const btnSi = document.getElementById('btnInternetSi');
            const btnNo = document.getElementById('btnInternetNo');
            const status = document.getElementById('internetStatus');
            
            btnSi.classList.remove('active');
            btnNo.classList.remove('active');
            
            if (valor === true) {{
                btnSi.classList.add('active');
                status.textContent = 'SI';
                status.className = 'internet-status si';
            }} else {{
                btnNo.classList.add('active');
                status.textContent = 'NO';
                status.className = 'internet-status no';
            }}
            
            guardarProgresoLocal();
            actualizarProgresoUnidad();
            
            // Guardar automáticamente al seleccionar
            guardarPreguntaIndividual(PREGUNTA_INTERNET);
        }}

        function guardarConsultoriosHabilitados() {{
            if (!unidadSeleccionada) {{
                alert('Primero seleccione una unidad medica');
                return;
            }}
            
            const input = document.getElementById('inputConsultoriosHabilitados');
            let valor = parseInt(input.value);
            
            if (isNaN(valor) || valor < 0) {{
                valor = null;
                input.value = '';
            }}
            if (valor > 50) {{
                valor = 50;
                input.value = 50;
                alert('El maximo permitido es 50');
            }}
            
            unidadSeleccionada[PREGUNTA_CONSULTORIOS_HABILITADOS] = valor;
            
            const status = document.getElementById('consultoriosHabilitadosStatus');
            if (valor !== null && !isNaN(valor)) {{
                status.textContent = valor + ' habilitados';
                if (valor === 0) {{
                    status.className = 'valor-actual cero';
                }} else {{
                    status.className = 'valor-actual';
                }}
            }} else {{
                status.textContent = 'PENDIENTE';
                status.className = 'valor-actual vacio';
            }}
            
            guardarProgresoLocal();
            actualizarProgresoUnidad();
        }}

        function cargarPreguntasIndividuales() {{
            if (!unidadSeleccionada) return;
            
            const internetValor = unidadSeleccionada[PREGUNTA_INTERNET];
            const btnSi = document.getElementById('btnInternetSi');
            const btnNo = document.getElementById('btnInternetNo');
            const statusInternet = document.getElementById('internetStatus');
            
            btnSi.classList.remove('active');
            btnNo.classList.remove('active');
            
            if (internetValor === true) {{
                btnSi.classList.add('active');
                statusInternet.textContent = 'SI';
                statusInternet.className = 'internet-status si';
            }} else if (internetValor === false) {{
                btnNo.classList.add('active');
                statusInternet.textContent = 'NO';
                statusInternet.className = 'internet-status no';
            }} else {{
                statusInternet.textContent = 'PENDIENTE';
                statusInternet.className = 'internet-status pendiente';
            }}
            
            const habilitadosValor = unidadSeleccionada[PREGUNTA_CONSULTORIOS_HABILITADOS];
            const input = document.getElementById('inputConsultoriosHabilitados');
            const statusHabilitados = document.getElementById('consultoriosHabilitadosStatus');
            
            if (habilitadosValor !== null && habilitadosValor !== undefined && !isNaN(habilitadosValor)) {{
                input.value = habilitadosValor;
                if (habilitadosValor === 0) {{
                    statusHabilitados.textContent = '0 habilitados';
                    statusHabilitados.className = 'valor-actual cero';
                }} else {{
                    statusHabilitados.textContent = habilitadosValor + ' habilitados';
                    statusHabilitados.className = 'valor-actual';
                }}
            }} else {{
                input.value = '';
                statusHabilitados.textContent = 'PENDIENTE';
                statusHabilitados.className = 'valor-actual vacio';
            }}
        }}

        function resetearPreguntasIndividuales() {{
            const btnSi = document.getElementById('btnInternetSi');
            const btnNo = document.getElementById('btnInternetNo');
            const statusInternet = document.getElementById('internetStatus');
            
            btnSi.classList.remove('active');
            btnNo.classList.remove('active');
            statusInternet.textContent = 'PENDIENTE';
            statusInternet.className = 'internet-status pendiente';
            
            const input = document.getElementById('inputConsultoriosHabilitados');
            const statusHabilitados = document.getElementById('consultoriosHabilitadosStatus');
            
            input.value = '';
            statusHabilitados.textContent = 'PENDIENTE';
            statusHabilitados.className = 'valor-actual vacio';
        }}

        // ==================== FUNCIONES DE CONSULTORIOS ====================
        function aplicarConsultorios() {{
            const input = document.getElementById('numConsultorios');
            let valor = parseInt(input.value);
            
            if (isNaN(valor) || valor < 0) {{
                valor = 0;
                input.value = 0;
            }}
            if (valor > 20) {{
                valor = 20;
                input.value = 20;
                alert('El maximo de consultorios permitido es 20');
            }}
            
            numConsultorios = valor;
            document.getElementById('consultoriosInfo').textContent = 'Consultorios configurados: ' + numConsultorios;
            
            if (unidadSeleccionada) {{
                mostrarUnidadTabla(unidadSeleccionada);
                actualizarProgresoUnidad();
                guardarProgresoLocal();
            }}
        }}

        function mostrarUnidadTabla(unidad) {{
            const tbody = document.getElementById('tableBody');
            const thead = document.getElementById('tableHead');
            tbody.innerHTML = '';
            
            const nombreUnidad = unidad.nombre_de_la_unidad || 'Sin nombre';
            const cluesUnidad = unidad.clues_imb || 'Sin CLUES';
            
            const claveUnidad = usuarioActual.email + '|' + unidad.clues_imb;
            const estaGuardada = filasGuardadasBD.has(claveUnidad);
            
            let headerRow = '<tr><th style="min-width: 280px; max-width: 350px;">Pregunta</th>';
            
            for (let i = 1; i <= numConsultorios; i++) {{
                headerRow += '<th style="min-width: 120px;">Consultorio ' + i + '</th>';
            }}
            
            headerRow += '</tr>';
            thead.innerHTML = headerRow;
            
            const preguntasFiltradas = columnasEquipamiento.filter(p => 
                p !== PREGUNTA_INTERNET && p !== PREGUNTA_CONSULTORIOS_HABILITADOS
            );
            
            for (let i = 0; i < preguntasFiltradas.length; i++) {{
                const pregunta = preguntasFiltradas[i];
                const row = tbody.insertRow();
                
                if (estaGuardada) {{
                    row.classList.add('fila-guardada-bd');
                }}
                
                const nombrePregunta = formatearNombreEquipo(pregunta);
                const preguntaKey = pregunta.toLowerCase().replace(/ /g, '_');
                
                const cellPregunta = row.insertCell(0);
                cellPregunta.innerHTML = '<span class="equipo-nombre">' + nombrePregunta + '</span>';
                cellPregunta.style.backgroundColor = estaGuardada ? '#c3e6cb' : '#f0f0f0';
                
                cellPregunta.addEventListener('mouseenter', function(e) {{
                    mostrarTooltip(e, 'Pregunta: ' + nombrePregunta);
                }});
                cellPregunta.addEventListener('mousemove', function(e) {{
                    mostrarTooltip(e, 'Pregunta: ' + nombrePregunta);
                }});
                cellPregunta.addEventListener('mouseleave', function() {{
                    ocultarTooltip();
                }});
                
                for (let j = 1; j <= numConsultorios; j++) {{
                    const cellConsultorio = row.insertCell();
                    const colConsultorio = preguntaKey + '_consultorio_' + j;
                    const valorConsultorio = unidad[colConsultorio] !== undefined ? unidad[colConsultorio] : null;
                    const filaOriginal = datosActuales.indexOf(unidad);
                    const campoConsultorio = crearCampoEquipamiento(valorConsultorio, filaOriginal, colConsultorio, nombrePregunta, j);
                    cellConsultorio.appendChild(campoConsultorio);
                    
                    cellConsultorio.addEventListener('mouseenter', function(e) {{
                        this.style.backgroundColor = '#fff3e0';
                        mostrarTooltip(e, nombrePregunta + ' (Consultorio ' + j + ')\\nUnidad: ' + nombreUnidad);
                    }});
                    cellConsultorio.addEventListener('mousemove', function(e) {{
                        mostrarTooltip(e, nombrePregunta + ' (Consultorio ' + j + ')\\nUnidad: ' + nombreUnidad);
                    }});
                    cellConsultorio.addEventListener('mouseleave', function() {{
                        this.style.backgroundColor = '';
                        ocultarTooltip();
                    }});
                }}
            }}
        }}

        function crearCampoEquipamiento(valorActual, filaOriginal, columna, nombrePregunta, numConsultorio) {{
            const container = document.createElement('div');
            container.className = 'campo-container';
            
            const valorMostrado = document.createElement('div');
            valorMostrado.className = 'valor-mostrado';
            
            const tieneValor = (valorActual !== null && valorActual !== undefined && !isNaN(valorActual) && valorActual !== '');
            const valorDisplay = tieneValor ? valorActual : 'PENDIENTE';
            
            valorMostrado.textContent = valorDisplay;
            
            if (!tieneValor) {{
                valorMostrado.classList.add('valor-vacio');
            }} else if (valorActual === 0) {{
                valorMostrado.classList.add('valor-cero');
            }}
            
            const claveNube = usuarioActual.email + '|' + unidadSeleccionada.clues_imb + '|' + columna;
            if (camposGuardadosNube.has(claveNube) && tieneValor) {{
                valorMostrado.classList.add('valor-guardado-nube');
            }}
            
            const iniciarEdicion = () => {{
                if (celdaEnEdicion !== null) {{
                    alert('Primero presione Enter para guardar el valor que esta editando');
                    return;
                }}
                
                celdaEnEdicion = container;
                container.innerHTML = '';
                
                const inputField = document.createElement('input');
                inputField.type = 'number';
                inputField.step = '1';
                inputField.value = (tieneValor && valorActual !== null) ? valorActual : '';
                inputField.placeholder = '0';
                inputField.className = 'input-edicion';
                
                // Botón Guardar oculto pero funcional
                const btnGuardar = document.createElement('button');
                btnGuardar.className = 'btn-guardar-individual';
                btnGuardar.textContent = 'Guardar';
                btnGuardar.id = 'btnGuardar' + columna.replace(/[^a-zA-Z0-9]/g, '');
                
                const finalizarEdicion = () => {{
                    const nuevoValor = inputField.value;
                    
                    if (nuevoValor === '') {{
                        datosActuales[filaOriginal][columna] = null;
                    }} else {{
                        let numero = parseInt(nuevoValor);
                        if (!isNaN(numero) && numero >= 0) {{
                            datosActuales[filaOriginal][columna] = numero;
                            unidadSeleccionada[columna] = numero;
                        }} else {{
                            alert('Ingrese un numero valido (0 o mayor)');
                            return;
                        }}
                    }}
                    
                    celdaEnEdicion = null;
                    guardarProgresoLocal();
                    
                    const valorFinal = datosActuales[filaOriginal][columna];
                    if (valorFinal !== null && valorFinal !== undefined && !isNaN(valorFinal) && valorFinal !== '') {{
                        guardarPreguntaIndividual(columna);
                    }}
                    
                    if (unidadSeleccionada) {{
                        mostrarUnidadTabla(unidadSeleccionada);
                    }}
                    actualizarProgresoUnidad();
                }};
                
                inputField.onkeypress = (e) => {{
                    if (e.key === 'Enter') {{
                        finalizarEdicion();
                    }}
                }};
                
                // También permitir guardar con el botón oculto (por si acaso)
                btnGuardar.onclick = finalizarEdicion;
                
                const inputContainer = document.createElement('div');
                inputContainer.style.display = 'flex';
                inputContainer.style.gap = '5px';
                inputContainer.style.alignItems = 'center';
                inputContainer.style.flexWrap = 'wrap';
                inputContainer.appendChild(inputField);
                inputContainer.appendChild(btnGuardar);
                
                container.appendChild(inputContainer);
                inputField.focus();
            }};
            
            valorMostrado.onclick = iniciarEdicion;
            container.appendChild(valorMostrado);
            
            return container;
        }}

        // ==================== FUNCIONES DE PROGRESO ====================
        function guardarProgresoLocal() {{
            if (estadoSeleccionado && datosActuales.length > 0 && usuarioActual.email) {{
                const clave = 'equipamiento_' + estadoSeleccionado + '_' + usuarioActual.email;
                const progreso = {{
                    entidad: estadoSeleccionado,
                    usuario: usuarioActual,
                    datos: datosActuales,
                    numConsultorios: numConsultorios,
                    camposGuardados: Array.from(camposGuardadosNube),
                    fecha_guardado: new Date().toISOString()
                }};
                localStorage.setItem(clave, JSON.stringify(progreso));
            }}
        }}

        function actualizarProgresoUnidad() {{
            if (!unidadSeleccionada) {{
                document.getElementById('progressInfo').innerHTML = '<p>Seleccione una unidad medica para ver el progreso</p>';
                return;
            }}
            
            const preguntasFiltradas = columnasEquipamiento.filter(p => 
                p !== PREGUNTA_INTERNET && p !== PREGUNTA_CONSULTORIOS_HABILITADOS
            );
            let completados = 0;
            let guardadosNube = 0;
            let totalCampos = preguntasFiltradas.length * numConsultorios;
            
            for (let c = 0; c < preguntasFiltradas.length; c++) {{
                const pregunta = preguntasFiltradas[c];
                const preguntaKey = pregunta.toLowerCase().replace(/ /g, '_');
                for (let j = 1; j <= numConsultorios; j++) {{
                    const colConsultorio = preguntaKey + '_consultorio_' + j;
                    const valor = unidadSeleccionada[colConsultorio];
                    if (valor !== null && valor !== undefined && !isNaN(valor) && valor !== '') {{
                        completados++;
                        const claveNube = usuarioActual.email + '|' + unidadSeleccionada.clues_imb + '|' + colConsultorio;
                        if (camposGuardadosNube.has(claveNube)) {{
                            guardadosNube++;
                        }}
                    }}
                }}
            }}
            
            const internetValor = unidadSeleccionada[PREGUNTA_INTERNET];
            const internetCompletado = (internetValor === true || internetValor === false);
            if (internetCompletado) {{
                completados++;
                const claveNube = usuarioActual.email + '|' + unidadSeleccionada.clues_imb + '|' + PREGUNTA_INTERNET;
                if (camposGuardadosNube.has(claveNube)) {{
                    guardadosNube++;
                }}
            }}
            totalCampos++;
            
            const habilitadosValor = unidadSeleccionada[PREGUNTA_CONSULTORIOS_HABILITADOS];
            const habilitadosCompletado = (habilitadosValor !== null && habilitadosValor !== undefined && !isNaN(habilitadosValor) && habilitadosValor !== '');
            if (habilitadosCompletado) {{
                completados++;
                const claveNube = usuarioActual.email + '|' + unidadSeleccionada.clues_imb + '|' + PREGUNTA_CONSULTORIOS_HABILITADOS;
                if (camposGuardadosNube.has(claveNube)) {{
                    guardadosNube++;
                }}
            }}
            totalCampos++;
            
            const porcentaje = totalCampos > 0 ? Math.round((completados / totalCampos) * 100) : 0;
            const porcentajeNube = totalCampos > 0 ? Math.round((guardadosNube / totalCampos) * 100) : 0;
            
            let badgeClass = porcentaje === 100 ? 'badge-success' : 'badge-warning';
            let badgeText = porcentaje === 100 ? 'COMPLETO' : 'PENDIENTE';
            
            document.getElementById('progressInfo').innerHTML = 
                '<strong>Progreso de llenado</strong>' +
                '<div style="background: #ddd; border-radius: 10px; margin-top: 5px;">' +
                    '<div style="background: {COLOR_SECUNDARIO}; width: ' + porcentaje + '%; height: 20px; border-radius: 10px; transition: width 0.3s;"></div>' +
                '</div>' +
                '<p style="margin-top: 5px;">' + completados + ' de ' + totalCampos + ' campos registrados (' + porcentaje + '%)</p>' +
                '<p style="font-size: 13px; color: #f57f17;"><i class="fas fa-cloud-upload-alt"></i> ' + guardadosNube + ' campos guardados en la nube (' + porcentajeNube + '%)</p>' +
                '<span class="badge ' + badgeClass + '">' + badgeText + '</span>' +
                '<p style="font-size: 13px; color: #666; margin-top: 5px;">' +
                    preguntasFiltradas.length + ' preguntas x ' + numConsultorios + ' consultorios + Internet + Consultorios Habilitados' +
                '</p>' +
                '<p style="font-size: 12px; color: #0d47a1; margin-top: 3px;">' +
                    '<i class="fas fa-info-circle"></i> Presione ENTER para guardar automáticamente cada valor.' +
                '</p>';
        }}

        function mostrarPopupUnidad() {{
            if (!unidadSeleccionada) {{
                alert('Seleccione una unidad medica primero');
                return;
            }}
            document.getElementById('popupClues').textContent = unidadSeleccionada.clues_imb || 'Sin CLUES';
            document.getElementById('popupNombre').textContent = unidadSeleccionada.nombre_de_la_unidad || 'Sin nombre';
            document.getElementById('popupCategoria').textContent = unidadSeleccionada.categoria_gerencial_ampliada || 'Sin categoria';
            document.getElementById('popupEntidad').textContent = estadoSeleccionado;
            document.getElementById('popupUsuario').textContent = usuarioActual.nombre + ' (' + usuarioActual.email + ')';
            document.getElementById('popupConsultorios').textContent = numConsultorios;
            
            const internetValor = unidadSeleccionada[PREGUNTA_INTERNET];
            let internetText = 'PENDIENTE';
            if (internetValor === true) internetText = 'SI';
            else if (internetValor === false) internetText = 'NO';
            document.getElementById('popupInternet').textContent = internetText;
            
            const habilitadosValor = unidadSeleccionada[PREGUNTA_CONSULTORIOS_HABILITADOS];
            let habilitadosText = 'PENDIENTE';
            if (habilitadosValor !== null && habilitadosValor !== undefined && !isNaN(habilitadosValor) && habilitadosValor !== '') {{
                habilitadosText = habilitadosValor;
            }}
            document.getElementById('popupConsultoriosHabilitados').textContent = habilitadosText;
            
            document.getElementById('popupUnidad').classList.add('active');
        }}

        // ==================== EVENTOS ====================
        document.getElementById('nombreInput').addEventListener('input', function() {{
            document.getElementById('errorMsg').style.display = 'none';
        }});
        
        document.getElementById('emailInput').addEventListener('input', function() {{
            document.getElementById('errorMsg').style.display = 'none';
        }});

        document.getElementById('loginModal').addEventListener('click', function(e) {{
            if (e.target === this) {{
                cerrarModal();
            }}
        }});

        document.getElementById('emailInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                validarDatos();
            }}
        }});
        
        document.getElementById('nombreInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                document.getElementById('emailInput').focus();
            }}
        }});
        
        document.addEventListener('mousemove', function(e) {{
            if (tooltip && tooltip.style.display === 'block') {{
                let left = e.pageX + 15;
                let top = e.pageY - 30;
                if (left + tooltip.offsetWidth > window.innerWidth) {{
                    left = e.pageX - tooltip.offsetWidth - 15;
                }}
                if (top < 0) {{
                    top = e.pageY + 20;
                }}
                tooltip.style.left = left + 'px';
                tooltip.style.top = top + 'px';
            }}
        }});

        document.getElementById('popupUnidad').addEventListener('click', function(e) {{
            if (e.target === this) {{
                cerrarPopup();
            }}
        }});

        document.getElementById('numConsultorios').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                aplicarConsultorios();
            }}
        }});
    </script>
</body>
</html>
'''


## salida

In [ ]:
# Guardar el archivo HTML
usuario = os.getlogin()
destino_html = fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\index.html"

try:
    with open(destino_html, "w", encoding="utf-8") as file:
        file.write(html_content)
    
   
    
    webbrowser.open(destino_html)
    
except Exception as e:
    
    destino_actual = os.path.join(os.getcwd(), "cuestionario_equipamiento_imss.html")
    with open(destino_actual, "w", encoding="utf-8") as file:
        file.write(html_content)
 
    webbrowser.open(destino_actual)